# Konu 4 — Temsil ve gömme vektörleri

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 3'te bir rengi üç sayıyla (R, G, B) gösterdik. Bir şeyi sayılarla göstermeye **temsil** diyoruz.
Bu konuda kelimeleri temsil edeceğiz ve Konu 1'deki kafe için soracağız: model "kafe"yi hangi kelimelere yakın buluyor?

## Isınma — elle iki eksenli harita

Konu 3'te bir renk üç sayıydı. Şimdi kelimeleri sayıya çevireceğiz; önce elle.
Bir mood-board düşün; iki ekseni olsun:

- **sıcaklık:** soğuk −1 … sıcak +1
- **enerji:** sakin −1 … canlı +1 (Konu 3'teki "sakin" / "enerjik" etiketleri gibi)

Altı kelimeye ikişer sayı verdik: ilki sıcaklık, ikincisi enerji.
Böyle bir sayı listesine **vektör** diyoruz: kahvenin vektörü 2 sayılı.

In [ ]:
kahve = [0.6, 0.4]
cay = [0.5, -0.4]
buz = [-0.9, 0.1]
deniz = [-0.5, -0.5]
ates = [0.9, 0.9]
kar = [-0.7, -0.2]

print(cay)

Çizerken her noktanın yanına adını yazacağız. Bunun için bir sözlük kuruyoruz: anahtar kelimenin adı, değer onun iki sayısı.

In [ ]:
harita = {
    "kahve": kahve,
    "çay": cay,
    "buz": buz,
    "deniz": deniz,
    "ateş": ates,
    "kar": kar,
}
print(harita["ateş"])

Haritayı çizelim. `for kelime in harita` sözlüğün anahtarlarını, yani kelimeleri sırayla verir.
`plt.scatter` bir nokta koyar, `plt.text` yanına kelimeyi yazar. `axhline` ve `axvline` gri
sıfır çizgilerini çizer: sağ taraf sıcak, üst taraf canlı.

In [ ]:
import matplotlib.pyplot as plt

for kelime in harita:
    nokta = harita[kelime]
    plt.scatter(nokta[0], nokta[1], color="#4a6fa5")
    plt.text(nokta[0], nokta[1], kelime)

plt.axhline(0, color="lightgray")
plt.axvline(0, color="lightgray")
plt.xlabel("soğuk — sıcak")
plt.ylabel("sakin — canlı")
plt.show()

Kahve ile ateş sağ üstte; buz, kar ve deniz solda toplandı: yakın duranlar birbirine benziyor. Eksenleri **biz** seçtik.

İki kelimenin ne kadar benzediğini tek sayıyla söyleyen **hazır bir araç** var:
`cosine_similarity` (Konu 3'teki scikit-learn'den). Onu küçük bir fonksiyona koyuyoruz.
Konu 3'teki `predict` gibi liste ister (`[a]`) ve cevabı tablo olarak verir. `[0]` ile satıra, bir `[0]` daha ile sayıya iniyoruz.
İçini bilmen gerekmiyor; verdiği sayıyı okuman yeter:

- **1**: aynı yön · **0**: ilgisiz · **−1**: zıt yön

**Yön:** haritanın ortasından (gri çizgilerin kesiştiği yerden) kelimeye bir ok çiz; okun baktığı taraf.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def benzerlik(a, b):
    sonuc = cosine_similarity([a], [b])
    satir = sonuc[0]
    return satir[0]

**Tahmin et:** kahve çaya mı daha çok benzer, buza mı? Hangisi eksi çıkar?

In [ ]:
print(benzerlik(kahve, cay))
print(benzerlik(kahve, buz))

Kahve de çay da sıcak tarafta: yaklaşık 0.3. Buz karşı tarafta: yaklaşık −0.77.

Hücrede `buz` yerine `ates` yaz: 0.98. Ateş kahveden çok daha sıcak ve canlı, ama **aynı yönde**.
Benzerlik okların uzunluğuna değil, **yönüne** bakar. Bundan sonra "yakın" dediğimizde bu sayıyı kastediyoruz: benzerlik büyükse iki kelime yakın.

Model de böyle bir harita kurar, ama eksenleri kendisi bulur: 2 değil, yüzlerce. Şimdi modele soralım.

## Adım 1 — Anahtarı oku

Konu 2'deki hücrenin aynısı. `anahtar.txt` bir üst klasörde, `gita3111`'in içinde durur.

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print("Anahtar okundu")

## Adım 2 — Bir kelimenin vektörü

Konu 2'deki isteğin aynısı, iki farkla:

- **Model** başka: metin yazan değil, metni sayılara çeviren bir model (Google'ın modeli; 100'den fazla dil biliyor).
- **Gövde** başka: `"text"` anahtarıyla bir liste gönderiyoruz.

In [ ]:
import requests

MODEL = "@cf/google/embeddinggemma-300m"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}
govde = {"text": ["kahve"]}

In [ ]:
cevap = requests.post(adres, headers=basliklar, json=govde)
print(cevap.status_code)

Yanıta katman katman iniyoruz. `data` bir liste, çünkü tek istekte birden çok metin
gönderebiliriz. Biz tek kelime gönderdik; ilki bizim.
Durum kodu 200 değilse bu hücre `TypeError` verir: 401 anahtar yanlış, 404 hesap kimliği yanlış, 429 kota doldu (Konu 2, Adım 5).

In [ ]:
yanit = cevap.json()
sonuc = yanit["result"]
vektorler = sonuc["data"]
vektor = vektorler[0]

**Tahmin et:** ısınmada "kahve" 2 sayıydı. Model kaç sayı verdi?

In [ ]:
print(len(vektor))
print(vektor[:5])

Bu da ısınmadaki gibi bir **vektör**, sadece çok daha uzun. Modelin kelimeyi vektöre çevirmesine **gömme** (embedding),
çıkan vektöre **gömme vektörü** diyoruz. Sayıların tek tek anlamı yok; anlam, kelimelerin birbirine göre nerede durduğunda.

## Adım 3 — Fonksiyona koy

Konu 2'deki `modele_sor`'un kardeşi: kelimeyi alır, vektörünü geri verir.
İçindeki satırlar Adım 2'nin aynısı. Tek ek, Konu 2'nin Adım 5'indeki kontrol: istek başarısızsa
hangi kelimede, hangi durum koduyla takıldığını yazar.

In [ ]:
def vektor_al(kelime):
    govde = {"text": [kelime]}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    if cevap.status_code != 200:
        print("Bir sorun var:", kelime, cevap.status_code)
    yanit = cevap.json()
    sonuc = yanit["result"]
    vektorler = sonuc["data"]
    return vektorler[0]

In [ ]:
vektor = vektor_al("çay")
print(len(vektor))

## Adım 4 — İki kelime ne kadar benzer?

**Çalıştırmadan önce tahmin et:** kahve hangisine daha çok benzer, çaya mı tipografiye mi?
Cevabı belli bir soruyla başlıyoruz: önce modelin sayılarını okumayı öğrenelim.

Isınmadaki `kahve` 2 sayıydı; şimdi aynı ad modelin verdiği vektörü tutuyor.
`benzerlik` fonksiyonu ikisiyle de çalışır. (`benzerlik` ısınmada tanımlandı; ısınmayı
atladıysan `NameError` alırsın.)

In [ ]:
kahve = vektor_al("kahve")
cay = vektor_al("çay")
tipografi = vektor_al("tipografi")

In [ ]:
print(benzerlik(kahve, cay))
print(benzerlik(kahve, tipografi))

Tahminin tuttu mu? Sayılara tek başına değil, **yan yana** bak: hangisi büyük?
Gerçek modelde ilgisiz kelimeler bile tam 0 çıkmayabilir; ısınmadaki 1 / 0 / −1 ölçeğiyle değil, kıyaslayarak oku.

## Adım 5 — 24 kelimelik sözlük

Dört grup, altışar kelime: renkler, duygular, tasarım, mekânlar.
Renklerde **bej** var: Konu 3'te kafe paleti için modele sorduğumuz renk.

In [ ]:
kelimeler = [
    "kırmızı", "mavi", "yeşil", "bej", "siyah", "beyaz",
    "huzur", "öfke", "neşe", "hüzün", "heyecan", "sakinlik",
    "tipografi", "logo", "afiş", "palet", "kontrast", "serif",
    "kafe", "kütüphane", "park", "ofis", "atölye", "sahne",
]
print(len(kelimeler))

Her kelime için ayrı bir istek gidiyor: 24 istek. Hücre çalışırken solunda `[*]` görürsün; bitince `24` yazar.
`24` yerine `Bir sorun var: ...` ve hata görürsen sözlük yarım kaldı: sorunu çözüp hücreyi yeniden çalıştır, sonra Adım 6'ya geç.
**Beklerken tahmin et:** "kafe" bu 24 kelimeden hangi üçüne en yakın çıkar? Not al, Adım 6'da bakacağız.

In [ ]:
vektorler_sozlugu = {}
for kelime in kelimeler:
    vektorler_sozlugu[kelime] = vektor_al(kelime)

print(len(vektorler_sozlugu))

## Adım 6 — "kafe"ye en yakın 3 kelime

Bu adımı **sen yaz**. Plan:

1. `skorlar` adında boş bir sözlük kur.
2. Sözlükteki her kelime için "kafe" ile o kelimenin benzerliğini hesapla, sonucu `skorlar[kelime]` olarak sakla.
3. Konu 1'deki `Counter` ile en büyükleri sırala. İpucu: `Counter`'a listeyi değil, `skorlar` sözlüğünü ver.

Bir kelimeye en yakın kelimelere **komşu** denir (Konu 3'teki model de rengin en yakın 5 komşusuna bakıyordu).
Aşağıdaki boş hücreye yaz. Altındaki cevaba bakmadan önce kendin dene.

Bir çözüm:

In [ ]:
hedef = "kafe"
hedef_vektor = vektorler_sozlugu[hedef]

skorlar = {}
for kelime in vektorler_sozlugu:
    vektor = vektorler_sozlugu[kelime]
    skorlar[kelime] = benzerlik(hedef_vektor, vektor)

print(len(skorlar))

Konu 1'de `Counter` listeyi kendisi sayıyordu. Sözlük verince saymaz, sözlükteki sayıları
olduğu gibi alır; `most_common` da onları büyükten küçüğe dizer.

In [ ]:
from collections import Counter

sirali = Counter(skorlar)
for kelime, skor in sirali.most_common(4):
    print(kelime, round(skor, 2))

**3 komşu istiyorduk, neden 4 yazdık?** İlk satıra bak. Adım 5'teki tahminin tuttu mu?

Konu 1'de müşteriler bu kafeyi **sessiz bir çalışma yeri** diye anlatmıştı. Model "kafe"yi kütüphaneye mi yakın buluyor, parka mı, sahneye mi?
Kütüphane uzakta kaldıysa "kafe" kelimesi tek başına bu markayı anlatmıyor demektir; sloganın başka bir kelimeye ihtiyacı var.

`hedef = "kafe"` yerine `"huzur"` yaz, iki hücreyi yeniden çalıştır: kafenin vermek istediği his hangi renge yakın?

## Adım 7 — Harita

Isınmadaki gibi bir harita istiyoruz. Ama her kelimede yüzlerce sayı var, kâğıt ise iki boyutlu.
**PCA** (Temel Bileşen Analizi, scikit-learn'den) yüzlerce sayıyı en çok şey anlatan 2 sayıya indirir.
Bir heykelin duvara düşen gölgesi gibi: PCA, gölgenin heykeli en iyi tanıttığı açıyı arar.

Önce bütün vektörleri bir listeye koyuyoruz:

In [ ]:
liste = []
for kelime in vektorler_sozlugu:
    liste.append(vektorler_sozlugu[kelime])

print(len(liste))

`fit`: Konu 3'teki gibi "önce örneklere bak". PCA bütün vektörlere bakıp en çok şey anlatan iki yönü bulur.
`n_components=2`: kaç sayıya ineceğimiz. Hücrenin altında `PCA(n_components=2)` yazan bir kutu çıkar: hazır demek.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(liste)

Şimdi her kelimeyi 2 sayıya çeviriyoruz ve ısınmadaki gibi bir sözlüğe koyuyoruz.
`transform`, Konu 3'teki `predict` gibi bir liste ister: `[vektor]`. Cevap da liste içinde gelir: `noktalar[0]`.

In [ ]:
model_haritasi = {}
for kelime in vektorler_sozlugu:
    vektor = vektorler_sozlugu[kelime]
    noktalar = pca.transform([vektor])
    model_haritasi[kelime] = noktalar[0]

print(model_haritasi["kafe"])

"kafe" artık ısınmadaki gibi 2 sayı (Konu 3'teki gibi virgülsüz yazılır). Çizim, ısınmadaki döngünün aynısı; yalnızca sözlüğün adı değişti.

In [ ]:
for kelime in model_haritasi:
    nokta = model_haritasi[kelime]
    plt.scatter(nokta[0], nokta[1], color="#4a6fa5")
    plt.text(nokta[0], nokta[1], kelime)

plt.show()

Yüzlerce sayıyı 2'ye indirdik. Harita bir **özet**; özetlerken bilgi kaybolur.
Haritada yan yana duran iki kelime, gerçekte o kadar yakın olmayabilir.
Eksenlerin adı da yok: ısınmada "sıcaklık" ve "enerji"yi biz koymuştuk, bu iki yönü PCA buldu.

## Adım 8 — Haritayı oku

- Dört grup (renkler, duygular, tasarım, mekânlar) ayrı kümeler oluşturdu mu?
- Kendi grubundan kaçan bir kelime var mı? Nereye gitmiş?
- Hangi renk duyguların kümesine en yakın? Bej nereye düştü?
- **huzur** ile **öfke** zıt anlamlı. Haritada uzak mı düştüler?

Haritaya güvenmeden asıl sayılara bakalım. **Tahmin et:** huzur, öfkeye mi daha yakın, sakinliğe mi?

In [ ]:
huzur = vektorler_sozlugu["huzur"]
ofke = vektorler_sozlugu["öfke"]
sakinlik = vektorler_sozlugu["sakinlik"]

print(benzerlik(huzur, ofke))
print(benzerlik(huzur, sakinlik))

Zıt anlamlı iki kelime bile yakın düşebilir: ikisi de duygulardan söz eden cümlelerde,
aynı bağlamda geçer. Modelin "benzer"i **eşanlamlı** demek değil, **benzer yerlerde geçen** demek.
Mood-board için modelden "huzur"a yakın kelimeler istersen, araya öfke de girebilir.

## Bonus — Kendi 5 kelimen

Listeye kendi beş kelimeni yaz, çalıştır. Sonra Adım 7'nin dört hücresini sırayla yeniden çalıştır.
Örnek listeye bak: "sessizlik" kafeye mi düştü, kütüphaneye mi?

In [ ]:
yeni_kelimeler = ["espresso", "minimal", "gürültü", "sessizlik", "poster"]
for kelime in yeni_kelimeler:
    vektorler_sozlugu[kelime] = vektor_al(kelime)

print(len(vektorler_sozlugu))